In [1]:
# import libraries
import numpy as np
import pickle
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
import concurrent.futures
import os
from adjustText import adjust_text

In [3]:
def visualize_heatmap(data, labels=[], label_sizes=[], cbar_label=r'$Log_{10} (Pr[i \to j])$', title="", suptitle="",  first_and_last_factor=1):
    # Expand first and last rows
    first_row_repeated = np.tile(data[0, :], (first_and_last_factor-1, 1))
    data = np.append(first_row_repeated, data, axis=0)
    first_col_repeated = np.tile(data[:, 0], (first_and_last_factor-1, 1))
    data = np.append(first_col_repeated.T, data, axis=1)
    last_row_repeated = np.tile(data[-1, :], (first_and_last_factor-1, 1))
    data = np.append(data, last_row_repeated, axis=0)
    last_col_repeated = np.tile(data[:, -1], (first_and_last_factor-1, 1))
    data = np.append(data, last_col_repeated.T, axis=1)

    
    fig = plt.figure(dpi=300)
    
    # Create the heatmap
    plt.imshow(data, cmap='jet', interpolation='nearest')
    
    # Add colorbar
    cbar = plt.colorbar()
    cbar.set_label(cbar_label)
    
    # Calculate tick positions (center of each group)
    tick_positions = []
    current_pos = 0
    
    for size in label_sizes:
        tick_positions.append(current_pos + size / 2 - 0.5)
        current_pos += size
    
    # Set ticks and labels
    plt.xticks(tick_positions, labels, rotation="vertical")
    plt.yticks(tick_positions, labels)
    
    # Add gridlines at label boundaries
    if len(label_sizes) > 1:
        grid_lines = [sum(label_sizes[:i]) - 0.5 for i in range(1, len(label_sizes))]
        for pos in grid_lines:
            plt.axhline(y=pos, color='black', linestyle='-', linewidth=0.5)
            plt.axvline(x=pos, color='black', linestyle='-', linewidth=0.5)
    
    # Set title and axis labels
    plt.suptitle(suptitle, fontsize=12)
    plt.title(title, fontsize=10)
    plt.xlabel("To")
    plt.ylabel("From")
    
    
    # Show the plot
    plt.show()

def labels_and_sizes(states, first_and_last_factor=5):

    labels = []
    label_sizes = []
    prev_label = ""
    cur_size = first_and_last_factor
    for i in range(len(states)):
        cur_label = states[i]    
        if cur_label == "nuc":
            labels.append(cur_label)
            prev_label = cur_label
            continue
        if cur_label == "cyt":
            labels.append(cur_label)
            label_sizes.append(cur_size)
            label_sizes.append(first_and_last_factor)
            prev_label = cur_label
            break
        
        cur_label = cur_label[:-3]
        
        if cur_label == prev_label:
            cur_size += 1
            continue
        label_sizes.append(cur_size)
        cur_size = 1
        labels.append(cur_label)
        prev_label = cur_label
    
    return labels, label_sizes

In [25]:
def generate_transition_matrix_from_multiple(sim_indexes, sim_times, file_name, step=1):
    #############
    # Load data #
    #############
    i_time_iterator = [(i, time) for i in sim_indexes for time in sim_times]

    def process_file(i_time):
        i, time = i_time
        print(f"{time}, {i} ", end="")
        with open(f"data/singles/{i}/{time}.pickle", "rb") as f:
            diffuser_trajectories = pickle.load(f)
        with open(f"data/singles/{i}/{time}-fgs.pickle", "rb") as f:
            fg_trajectories = pickle.load(f)
        return categorize_diffusers_over_time(diffuser_trajectories, fg_trajectories, 1, step=step)

    arrays = []
    num_processes = len(os.sched_getaffinity(0))
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
        results = list(executor.map(process_file, i_time_iterator))

    arrays.extend(results)
    all = np.concatenate(arrays, axis=0)

    #######################################
    # Generate and save transition matrix #
    #######################################
    tm, states = generate_transition_matrix(all[:,0,:], init_1=True, symmetrize=True)
    with open(file_name, "wb") as f:
        pickle.dump((tm, states), f)


In [ ]:
generate_transition_matrix_from_multiple(
    sim_indexes = range(1, 51),
    sim_times = ["150-160", "160-170", "170-180"],
    file_name = "data/transition_matrices/interactions-150-180-symmetric.pickle"
    )

In [26]:
for step in [2, 4, 8, 16, 32, 64]:
    generate_transition_matrix_from_multiple(
    sim_indexes = range(1, 51),
    sim_times = ["150-160", "160-170", "170-180"],
    file_name = f"data/transition_matrices/interactions-150-180-symmetric-{100*step}ns.pickle",
    step = step
    )

150-160, 1 160-170, 1 170-180, 1 150-160, 2 160-170, 2 170-180, 2 150-160, 3 160-170, 3 170-180, 3 150-160, 4 160-170, 4 170-180, 4 150-160, 5 160-170, 5 170-180, 5 150-160, 6 160-170, 6 170-180, 6 150-160, 7 160-170, 7 170-180, 7 150-160, 8 160-170, 8 170-180, 8 150-160, 9 160-170, 9 170-180, 9 150-160, 10 160-170, 10 170-180, 10 150-160, 11 160-170, 11 170-180, 11 150-160, 12 160-170, 12 170-180, 12 150-160, 13 160-170, 13 170-180, 13 150-160, 14 160-170, 14 170-180, 14 150-160, 15 160-170, 15 170-180, 15 150-160, 16 160-170, 16 170-180, 16 150-160, 17 160-170, 17 170-180, 17 150-160, 18 160-170, 18 170-180, 18 150-160, 19 160-170, 19 170-180, 19 150-160, 20 160-170, 20 170-180, 20 150-160, 21 160-170, 21 170-180, 21 150-160, 22 160-170, 22 170-180, 22 150-160, 23 160-170, 23 170-180, 23 150-160, 24 160-170, 24 170-180, 24 150-160, 25 160-170, 25 170-180, 25 150-160, 26 160-170, 26 170-180, 26 150-160, 27 160-170, 27 170-180, 27 150-160, 28 160-170, 28 170-180, 28 150-160, 29 160-170

In [33]:
def load_and_visualize_heatmap(path, suptitle="", cbar_label=r'$Log_{10} (Pr[i \to j])$ over 100 ns', transform_transition_dt=-1, log_10=False):

    with open(path, "rb") as f:
        tm, states = pickle.load(f)

    with open(f"data/anchor_coordinates.pickle", "rb") as f:
        anchor_coordinates = pickle.load(f)
        
    # Sort anchor coordinates by Z
    anchor_coordinates = dict(sorted(anchor_coordinates.items(), key=lambda x: x[1][2]))
    new_states = ["nuc"] + list(anchor_coordinates.keys()) + ["cyt"]
    tm = reorder_transition_matrix(tm, states, new_states)
    if transform_transition_dt != -1: tm = infinitesimal_generator(tm, dt=transform_transition_dt)
    if log_10: tm = np.log10(tm)

    first_and_last_factor = 20
    labels, label_sizes = labels_and_sizes(new_states, first_and_last_factor=first_and_last_factor)
    visualize_heatmap(tm,
                    suptitle=suptitle,
                    title="",
                    labels=labels,
                    label_sizes=label_sizes,
                    cbar_label=cbar_label,
                    first_and_last_factor=first_and_last_factor)
    


In [ ]:
for step in [1, 2, 4, 8, 16, 32, 64]:
    load_and_visualize_heatmap(f"data/transition_matrices/interactions-150-180-symmetric-{100*step}ns.pickle",
                               suptitle=f"Interaction transition matrix, 150-180ns, 50 sims",
                               cbar_label=r'$Log_{10} (Pr[i \to j])$ over ' + str(step * 100) + " ns",
                               log_10=True)

In [ ]:
for step in [1, 2, 4, 8, 16]: #, 32, 64]: # above 16 doesn't work
    load_and_visualize_heatmap(f"data/transition_matrices/interactions-150-180-symmetric-{100*step}ns.pickle",
                               suptitle=f"Transition rate matrix, 150-180ns, 50 sims, calculated from transition matrix {step * 100} ns steps",
                               cbar_label=r'$\frac{1}{ns}$', # todo not log10?
                               transform_transition_dt=step * 100)

In [1]:
for step in [1, 2, 4, 8, 16]: #, 32, 64]: # above 16 doesn't work
    load_and_visualize_heatmap(f"data/transition_matrices/interactions-150-180-symmetric-{100*step}ns.pickle",
                               suptitle=f"Transition rate matrix, 150-180ns, 50 sims, calculated from transition matrix {step * 100} ns steps",
                               cbar_label=r'$Log_{10} (\frac{1}{ns})$', # todo not log10?
                               transform_transition_dt=step * 100,
                               log_10=True)

NameError: name 'load_and_visualize_heatmap' is not defined

In [2]:
def visualize_graph_opposite_spokes(matrix_path, step=1, title=""):
    # Load transition matrix and states
    with open(matrix_path, "rb") as f:
        tm, states = pickle.load(f)

    # Load anchor coordinates
    with open(f"data/anchor_coordinates.pickle", "rb") as f:
        anchor_coordinates = pickle.load(f)
    
    # Sort anchor coordinates by Z and reorder transition matrix
    anchor_coordinates = dict(sorted(anchor_coordinates.items(), key=lambda x: x[1][2]))
    new_states = ["nuc"] + list(anchor_coordinates.keys()) + ["cyt"]
    tm = reorder_transition_matrix(tm, states, new_states)

    # Extract nodes coordinates and labels (taking every 4th point starting from index 1)
    # This appears to select specific anchor points from the dataset
    node_indices = [i for i in range(len(anchor_coordinates)) if (i - 2) % 4 == 0]
    nodes = np.array([list(anchor_coordinates.values())[i][1:3] for i in node_indices])
    labels = [list(anchor_coordinates.keys())[i][:-3] for i in node_indices]
    
    # Extract edges and their weights from the transition matrix
    edges = []
    edge_weights = []
    num_nodes = len(nodes)
    
    # Map from node_indices to corresponding positions in the transition matrix
    # Adding 1 accounts for the "nuc" state at index 0
    tm_indices = [i + 1 for i in node_indices]
    
    # Create edges between nodes with non-zero transition probabilities
    for i, tm_i in enumerate(tm_indices):
        for j, tm_j in enumerate(tm_indices):
            if tm[tm_i, tm_j] > 0.0:
                edges.append((nodes[i], nodes[j]))
                edge_weights.append(tm[tm_i, tm_j])
    
    # Convert to numpy arrays
    edges = np.array(edges)
    edge_weights = np.array(edge_weights)
    
    # Normalize edge weights to [0,1] for opacity
    if len(edge_weights) > 1:  # Avoid division by zero if only one edge
        normalized_weights = (edge_weights - np.min(edge_weights)) / (np.max(edge_weights) - np.min(edge_weights))
    else:
        normalized_weights = np.array([1.0])

    # Visualization
    fig = plt.figure(dpi=250)
    ax = fig.add_subplot(111)
    
    # Extract x and y coordinates
    x = nodes[:, 0]
    y = nodes[:, 1]
    
    # Plot nodes
    ax.scatter(x, y, alpha=0.2, s=100, color="blue")
    
    # Plot edges with opacity proportional to transition probability
    for i, (start_point, end_point) in enumerate(edges):
        ax.plot([start_point[0], end_point[0]], 
                [start_point[1], end_point[1]], 
                color="gray", 
                alpha=normalized_weights[i])
    
    # Add text labels
    texts = []
    for xi, yi, label in zip(x, y, labels):
        texts.append(ax.text(xi, yi, label, fontsize=6))

    # Adjust the position of the texts to minimize overlap and add arrows
    adjust_text(texts, 
                only_move={'points': 'xy', 'texts': 'xy'}
                # arrowprops=dict(arrowstyle='->', color='red')
                )

    # Set plot layout and display
    plt.tight_layout()
    plt.ylim(-30, 30)
    ax.set_aspect('equal')
    plt.title(title)
    
    return plt.show()

In [ ]:
for t in ["100ns", "200ns", "400ns", "800ns", "1600ns", "3200ns", "6400ns"]:
    visualize_graph_opposite_spokes(f"data/transition_matrices/interactions-150-180-symmetric-{t}.pickle", title=f"{t} steps")

In [8]:
def visualize_graph_three_spokes(matrix_path, title=""):
    rotation_angle_degrees=-52
    
    # Load transition matrix and states
    with open(matrix_path, "rb") as f:
        tm, states = pickle.load(f)

    # Load anchor coordinates
    with open(f"data/anchor_coordinates.pickle", "rb") as f:
        anchor_coordinates = pickle.load(f)
    
    # Sort anchor coordinates by Z and reorder transition matrix
    anchor_coordinates = dict(sorted(anchor_coordinates.items(), key=lambda x: x[1][2]))
    new_states = ["nuc"] + list(anchor_coordinates.keys()) + ["cyt"]
    tm = reorder_transition_matrix(tm, states, new_states)

    # Extract nodes coordinates and labels (taking every 4th point starting from index 1)
    # This appears to select specific anchor points from the dataset
    node_indices = [i for i in range(len(anchor_coordinates)) if ((i - 1) % 8 == 0) or ((i - 2) % 8 == 0) or ((i) % 8 == 0)]
    
    # Get all three coordinates (x, y, z)
    raw_coords = np.array([list(anchor_coordinates.values())[i] for i in node_indices])
    labels = [list(anchor_coordinates.keys())[i][:-3] for i in node_indices]
    
    # Convert angle to radians
    angle_rad = np.radians(rotation_angle_degrees)
    
    # Create rotated coordinates
    # Apply rotation transformation around z-axis
    # [x', y'] = [x*cos(θ) - y*sin(θ), x*sin(θ) + y*cos(θ)]
    rotated_x = raw_coords[:, 0] * np.cos(angle_rad) - raw_coords[:, 1] * np.sin(angle_rad)
    rotated_y = raw_coords[:, 0] * np.sin(angle_rad) + raw_coords[:, 1] * np.cos(angle_rad)
    
    # Combine rotated x,y with original z to form nodes
    # Use rotated_y and original z for the visualization
    nodes = np.column_stack((rotated_y, raw_coords[:, 2]))
    
    # Extract edges and their weights from the transition matrix
    edges = []
    edge_weights = []
    num_nodes = len(nodes)
    
    # Map from node_indices to corresponding positions in the transition matrix
    # Adding 1 accounts for the "nuc" state at index 0
    tm_indices = [i + 1 for i in node_indices]
    
    # Create edges between nodes with non-zero transition probabilities
    for i, tm_i in enumerate(tm_indices):
        for j, tm_j in enumerate(tm_indices):
            if tm[tm_i, tm_j] > 0.0:
                edges.append((nodes[i], nodes[j]))
                edge_weights.append(tm[tm_i, tm_j])
    
    # Convert to numpy arrays
    edges = np.array(edges)
    edge_weights = np.array(edge_weights)
    
    # Normalize edge weights to [0,1] for opacity
    if len(edge_weights) > 1:  # Avoid division by zero if only one edge
        normalized_weights = (edge_weights - np.min(edge_weights)) / (np.max(edge_weights) - np.min(edge_weights))
    else:
        normalized_weights = np.array([1.0])

    # Visualization
    fig = plt.figure(dpi=250)
    ax = fig.add_subplot(111)
    
    # Extract x and y coordinates
    x = nodes[:, 0]
    y = nodes[:, 1]
    
    # Plot nodes
    x1 = [xi for i, xi in enumerate(x) if i % 3 == 0]
    y1 = [yi for i, yi in enumerate(y) if i % 3 == 0]
    x2 = [xi for i, xi in enumerate(x) if i % 3 == 1]
    y2 = [yi for i, yi in enumerate(y) if i % 3 == 1]
    x3 = [xi for i, xi in enumerate(x) if i % 3 == 2]
    y3 = [yi for i, yi in enumerate(y) if i % 3 == 2]
    ax.scatter(x1, y1, alpha=0.2, s=100, color="red")
    ax.scatter(x2, y2, alpha=0.2, s=100, color="green")
    ax.scatter(x3, y3, alpha=0.2, s=100, color="blue")
    
    # Plot edges with opacity proportional to transition probability
    for i, (start_point, end_point) in enumerate(edges):
        ax.plot([start_point[0], end_point[0]], 
                [start_point[1], end_point[1]], 
                color="gray", 
                alpha=normalized_weights[i])
    
    # Add text labels
    texts = []
    for xi, yi, label in zip(x, y, labels):
        texts.append(ax.text(xi, yi, label, fontsize=6))

    # Adjust the position of the texts to minimize overlap and add arrows
    adjust_text(texts, 
                only_move={'points': 'xy', 'texts': 'xy'}
                # arrowprops=dict(arrowstyle='->', color='red')
                )

    # Set plot layout and display
    plt.title(title)
    plt.tight_layout()
    plt.ylim(-30, 30)
    ax.set_aspect('equal')
    
    return plt.show()

# this is looking from the outside

In [ ]:
for t in ["100ns", "200ns", "400ns", "800ns", "1600ns", "3200ns", "6400ns"]:
    visualize_graph_three_spokes(f"data/transition_matrices/interactions-150-180-symmetric-{t}.pickle", title=f"{t} steps")